# BM25 Pipeline — Sinh tập 200 cặp (query, product) để Annotate

**Mục tiêu:** Dùng BM25 để stratified sampling, tạo tập ~200 cặp (query, product) đại diện cho 3 tier:
- **Tier 1 (40%):** High BM25 score — ca dễ, annotators nên đồng thuận
- **Tier 2 (50%):** Mid score + edge cases (màu sắc mismatch, từ hiếm) — ca khó, bộc lộ kẽ hở guideline
- **Tier 3 (10%):** Random — kiểm chứng tính khách quan

**Schema gán nhãn:** `exact` / `partial` / `irrelevant`

---
**Cách dùng:**
1. Upload `metadata_resized.json` lên Colab (hoặc Google Drive)
2. Chạy từng cell theo thứ tự
3. File output `annotation_pairs.xlsx` sẽ được tải về tự động

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Bước 0 — Cài đặt thư viện

In [2]:
!pip install rank_bm25 underthesea openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.5 MB/s eta 0:00:00


## Bước 1 — Upload dữ liệu

> Chạy cell này rồi upload 2 file:
> 1. `metadata_resized.json` (dữ liệu sản phẩm)
> 2. `query_list_200.xlsx` (danh sách 200 query cần dùng)
>
> Nếu đã có sẵn trên Google Drive, bỏ qua cell này và sửa đường dẫn `DATA_PATH` / `QUERY_LIST_PATH` ở Bước 2.

In [ ]:
from google.colab import files
uploaded = files.upload()
# Upload cả metadata_resized.json và query_list_200.xlsx
# Sau khi upload xong, tên file sẽ hiện ra bên dưới

## Bước 2 — Config & Load dữ liệu

In [3]:
import json
import random
import math
import re
import unicodedata
from collections import defaultdict

# ============================================================
# CONFIG — chỉnh ở đây nếu cần
# ============================================================
DATA_PATH = '/content/drive/MyDrive/KL/Data/metadata_resized.json'   # đường dẫn file JSON
QUERY_LIST_PATH = '/content/query_list_200.xlsx'   # đường dẫn file danh sách query (200 query)
                                           # nếu upload trực tiếp ở Bước 1 thì để tên file như trên;
                                           # nếu để trên Drive thì đổi thành đường dẫn đầy đủ
TARGET_PAIRS = 200                    # tổng số cặp output
RANDOM_SEED = 42

# Phân bổ 3 tier
TIER1_RATIO = 0.40   # high score  → 80 cặp
TIER2_RATIO = 0.50   # edge cases  → 100 cặp
TIER3_RATIO = 0.10   # random      → 20 cặp

# BM25 params
BM25_K1 = 1.5
BM25_B  = 0.75

# Số sản phẩm lấy mỗi query cho mỗi tier
TOP_K_TIER1 = 5    # lấy top-5 per query
MID_K_TIER2 = 8    # lấy 8 sản phẩm mid-range per query
# ============================================================

random.seed(RANDOM_SEED)

with open(DATA_PATH, encoding='utf-8') as f:
    products = json.load(f)

print(f"✅ Loaded {len(products)} products")
print(f"   Shops: {set(p['shop'] for p in products)}")
print(f"   Categories: {set(p.get('category','?') for p in products)}")

✅ Loaded 3331 products
   Shops: {'aristino', 'icondenim', 'henstore', 'johnhenry'}
   Categories: {'quan', 'ao', 'phu-kien'}


## Bước 3 — Tiền xử lý văn bản (Text Preprocessing)

In [4]:
# Stopwords tiếng Việt — giữ lại từ mang nghĩa thời trang
STOPWORDS = {
    'và', 'của', 'là', 'có', 'trong', 'với', 'các', 'những', 'được',
    'cho', 'từ', 'này', 'đó', 'cũng', 'khi', 'về', 'theo', 'tại',
    'để', 'bởi', 'hoặc', 'nếu', 'thì', 'mà', 'hay', 'vì', 'nên',
    'như', 'sẽ', 'đã', 'đang', 'rất', 'một', 'không', 'hơn', 'hơn',
    'lên', 'xuống', 'ra', 'vào', 'đi', 'lại', 'giúp', 'tạo', 'mang',
    'sản', 'phẩm', 'hàng'
    # LƯU Ý: KHÔNG loại 'dài', 'ngắn', 'xanh', 'trắng', 'đen', 'đỏ'
    # vì đây là từ ngữ nghĩa quan trọng trong thời trang
}

# Normalize màu sắc: map tiếng Anh → tiếng Việt phổ biến
COLOR_NORMALIZE = {
    'blue': 'xanh', 'navy': 'xanh navy', 'black': 'đen',
    'white': 'trắng', 'grey': 'xám', 'gray': 'xám',
    'green': 'xanh lá', 'red': 'đỏ', 'brown': 'nâu',
    'beige': 'be', 'pink': 'hồng', 'yellow': 'vàng',
    'orange': 'cam', 'purple': 'tím', 'light': 'nhạt',
    'dark': 'đậm', 'solid': 'trơn', 'ora': 'cam',
}

def normalize_text(text: str) -> str:
    """Chuẩn hóa văn bản: lowercase, bỏ ký tự đặc biệt, normalize màu."""
    if not text:
        return ''
    text = text.lower()
    # Normalize màu tiếng Anh
    for en, vi in COLOR_NORMALIZE.items():
        text = re.sub(rf'\b{en}\b', vi, text)
    # Bỏ ký tự đặc biệt, giữ chữ và số và khoảng trắng
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text: str) -> list:
    """Tách từ đơn giản (split by space) sau khi normalize.
    Với production: thay bằng underthesea.word_tokenize(text)"""
    tokens = normalize_text(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

# Thử nghiệm
sample = products[0].get('text_document', '')
tokens = tokenize(sample)
print(f"Sample text_document (first 100 chars): {sample[:100]}")
print(f"Tokenized ({len(tokens)} tokens): {tokens[:20]}")

Sample text_document (first 100 chars): Áo Sơ Mi Dài Tay Nam Aristino Regular Fit ALS581S0H2 | Áo Sơ mi dài tay | ao ao-so-mi | màu: Xanh so
Tokenized (219 tokens): ['áo', 'sơ', 'mi', 'dài', 'tay', 'nam', 'aristino', 'regular', 'fit', 'als581s0h2', 'áo', 'sơ', 'mi', 'dài', 'tay', 'ao', 'ao', 'so', 'mi', 'màu']


## Bước 4 — Xây dựng BM25 Index

In [5]:
from rank_bm25 import BM25Okapi

# Tạo document text cho mỗi sản phẩm
# Chiến lược field boosting: lặp tên sản phẩm 3x, tags 2x, description 1x
def build_doc_text(product: dict) -> str:
    name = product.get('name', '')
    tags = ' '.join(product.get('tags', []))
    colors = ' '.join(product.get('colors', []))
    category = product.get('raw_product_type', '')
    text_doc = product.get('text_document', '')

    # Boost: name x3, category+colors+tags x2, full text x1
    boosted = f"{name} {name} {name} {category} {colors} {tags} {tags} {text_doc}"
    return boosted

print("Building BM25 index...")
corpus_texts = [build_doc_text(p) for p in products]
tokenized_corpus = [tokenize(t) for t in corpus_texts]

bm25 = BM25Okapi(tokenized_corpus, k1=BM25_K1, b=BM25_B)

# Tính IDF cho từng token (để dùng ở Tier 2)
N = len(products)
df = defaultdict(int)
for doc_tokens in tokenized_corpus:
    for token in set(doc_tokens):
        df[token] += 1

def idf(token):
    return math.log((N - df[token] + 0.5) / (df[token] + 0.5) + 1)

print(f"✅ BM25 index built on {N} products")
print(f"   Vocabulary size: {len(df)} unique tokens")

# Ví dụ IDF của một số token
example_tokens = ['xanh', 'polo', 'linen', 'regular', 'áo', 'quần', 'jacquard']
print("\nIDF examples:")
for tok in example_tokens:
    print(f"  '{tok}': IDF = {idf(tok):.3f}, DF = {df.get(tok, 0)}")

Building BM25 index...
✅ BM25 index built on 3331 products
   Vocabulary size: 7315 unique tokens

IDF examples:
  'xanh': IDF = 0.783, DF = 1523
  'polo': IDF = 1.475, DF = 762
  'linen': IDF = 3.814, DF = 73
  'regular': IDF = 1.260, DF = 945
  'áo': IDF = 0.357, DF = 2330
  'quần': IDF = 0.530, DF = 1961
  'jacquard': IDF = 2.856, DF = 191


## Bước 5 — Load danh sách Query (từ `query_list_200.xlsx`)

> Đọc 200 query thực tế từ file `query_list_200.xlsx` (sheet `Query List`, cột `Query`).
> File này bao phủ đủ các nhóm sản phẩm (áo sơ mi, áo polo, quần âu/kaki, quần jean/short/jogger,
> phụ kiện, phong cách tổng hợp...) và nhiều loại intent (màu, fit, chất liệu, họa tiết, dịp mặc...).
>
> Nếu muốn quay lại dùng list query viết tay, có thể gán thẳng `SAMPLE_QUERIES = [...]` thay cho đoạn code bên dưới.

In [6]:
import openpyxl as _openpyxl

def load_queries_from_excel(path: str, sheet_name: str = 'Query List') -> list:
    """Đọc cột 'Query' từ file query_list_200.xlsx, bỏ qua các dòng tiêu đề nhóm (── ... ──) và dòng trống."""
    wb_q = _openpyxl.load_workbook(path, data_only=True)
    ws_q = wb_q[sheet_name] if sheet_name in wb_q.sheetnames else wb_q.active

    header = [c.value for c in ws_q[1]]
    col_query = header.index('Query') if 'Query' in header else 2  # fallback: cột C

    queries, meta = [], []
    for row in ws_q.iter_rows(min_row=2, values_only=True):
        q = row[col_query]
        if q is None or not str(q).strip():
            continue  # bỏ dòng ngăn nhóm (STT=None, Query=None) hoặc dòng trống
        q = str(q).strip()
        if q not in queries:  # tránh trùng lặp
            queries.append(q)
            meta.append({
                'group': row[1] if len(row) > 1 else None,
                'intent': row[3] if len(row) > 3 else None,
            })
    return queries, meta

SAMPLE_QUERIES, QUERY_META = load_queries_from_excel(QUERY_LIST_PATH)

print(f"✅ Loaded {len(SAMPLE_QUERIES)} queries from '{QUERY_LIST_PATH}'")
print("\nQuery list (10 đầu tiên):")
for i, q in enumerate(SAMPLE_QUERIES[:10]):
    print(f"  [{i+1:02d}] {q}")
print("  ...")

from collections import Counter
group_counts = Counter(m['group'] for m in QUERY_META if m['group'])
print("\nPhân bổ theo nhóm:")
for g, c in group_counts.most_common():
    print(f"  {g}: {c}")

✅ Loaded 200 queries from '/content/query_list_200.xlsx'

Query list (10 đầu tiên):
  [01] áo sơ mi slim fit nam
  [02] áo sơ mi regular fit dài tay
  [03] áo sơ mi perfect fit công sở
  [04] áo sơ mi dáng ôm công sở
  [05] áo sơ mi dáng vừa không nhăn
  [06] áo sơ mi trắng dài tay nam
  [07] áo sơ mi đen dài tay
  [08] áo sơ mi xanh navy công sở
  [09] áo sơ mi xám dài tay
  [10] áo sơ mi be nam
  ...

Phân bổ theo nhóm:
  ÁO SƠ MI: 35
  ÁO POLO: 25
  QUẦN ÂU & KAKI: 25
  PHONG CÁCH TỔNG HỢP: 25
  QUẦN JEAN & SHORT & JOGGER: 20
  PHỤ KIỆN — THẮT LƯNG & CÀ VẠT: 20
  PHỤ KIỆN — GIÀY, TÚI, NÓN, VỚ: 20
  ÁO THUN: 15
  ÁO KHOÁC & BLAZER: 15


## Bước 6 — Stratified Sampling (3 Tiers)

In [7]:
import numpy as np

# ============================================================
# Hàm score và lấy ranked list cho một query
# ============================================================
def score_query(query: str):
    """Trả về list (score, product_idx) đã sort giảm dần."""
    tokens = tokenize(query)
    if not tokens:
        return []
    scores = bm25.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: -x[1])
    return ranked  # [(idx, score), ...]

def query_token_idf(query: str):
    """Trả về IDF trung bình của các token trong query."""
    tokens = tokenize(query)
    if not tokens:
        return 0
    return sum(idf(t) for t in tokens) / len(tokens)

# ============================================================
# TIER 1: High score — top-K per query
# ============================================================
tier1_pairs = []  # list of {query, product_idx, bm25_score, tier}
seen_pairs = set()

for query in SAMPLE_QUERIES:
    ranked = score_query(query)
    count = 0
    for idx, score in ranked:
        if score <= 0:
            break
        key = (query, idx)
        if key not in seen_pairs:
            tier1_pairs.append({'query': query, 'product_idx': idx,
                                 'bm25_score': round(score, 4), 'tier': 'T1_high'})
            seen_pairs.add(key)
            count += 1
        if count >= TOP_K_TIER1:
            break

# Shuffle và lấy đủ quota T1
random.shuffle(tier1_pairs)
n_tier1 = int(TARGET_PAIRS * TIER1_RATIO)
tier1_final = tier1_pairs[:n_tier1]
for p in tier1_final:
    seen_pairs.add((p['query'], p['product_idx']))

print(f"Tier 1 (high score): {len(tier1_final)} pairs (target {n_tier1})")

# ============================================================
# TIER 2: Edge cases — mid score + high IDF tokens
# ============================================================
# Ưu tiên:
#   2a. Mid-range BM25 score (rank 20-60)
#   2b. Query có IDF cao (từ hiếm, chuyên ngành)
#   2c. Màu sắc mismatch (query màu tiếng Việt, product ghi màu tiếng Anh)

COLOR_VI_TOKENS = {'xanh', 'trắng', 'đen', 'đỏ', 'nâu', 'xám', 'hồng', 'vàng', 'cam', 'tím', 'be'}
COLOR_EN_TOKENS = {'blue', 'navy', 'black', 'white', 'grey', 'gray', 'green', 'red',
                   'brown', 'beige', 'pink', 'yellow', 'orange', 'purple'}

def has_color_mismatch(query: str, product: dict) -> bool:
    """True nếu query dùng màu tiếng Việt nhưng product ghi màu tiếng Anh (hoặc ngược lại)."""
    q_tokens = set(tokenize(query))
    q_has_vi_color = bool(q_tokens & COLOR_VI_TOKENS)
    product_color_text = ' '.join(product.get('colors', [])).lower()
    prod_has_en_color = any(c in product_color_text for c in COLOR_EN_TOKENS)
    return q_has_vi_color and prod_has_en_color

tier2_candidates = []

for query in SAMPLE_QUERIES:
    ranked = score_query(query)
    q_idf = query_token_idf(query)
    n = len(ranked)

    # Mid range: rank 20% đến 70% của toàn bộ scored products
    scored = [(idx, score) for idx, score in ranked if score > 0]
    if len(scored) < 10:
        continue
    lo = int(len(scored) * 0.20)
    hi = int(len(scored) * 0.70)
    mid_range = scored[lo:hi]

    count = 0
    for idx, score in mid_range:
        key = (query, idx)
        if key in seen_pairs:
            continue
        product = products[idx]
        color_mismatch = has_color_mismatch(query, product)
        # Priority score: ưu tiên color mismatch và query IDF cao
        priority = q_idf + (2.0 if color_mismatch else 0)
        tier2_candidates.append({
            'query': query,
            'product_idx': idx,
            'bm25_score': round(score, 4),
            'tier': 'T2_edge',
            'color_mismatch': color_mismatch,
            'query_idf': round(q_idf, 3),
            'priority': priority
        })
        count += 1
        if count >= MID_K_TIER2:
            break

# Sort theo priority, lấy đủ quota T2
tier2_candidates.sort(key=lambda x: -x['priority'])
n_tier2 = int(TARGET_PAIRS * TIER2_RATIO)
tier2_final = []
for cand in tier2_candidates:
    key = (cand['query'], cand['product_idx'])
    if key not in seen_pairs:
        tier2_final.append(cand)
        seen_pairs.add(key)
    if len(tier2_final) >= n_tier2:
        break

print(f"Tier 2 (edge cases): {len(tier2_final)} pairs (target {n_tier2})")
print(f"  - Color mismatch cases: {sum(1 for p in tier2_final if p.get('color_mismatch'))}")

# ============================================================
# TIER 3: Random
# ============================================================
n_tier3 = TARGET_PAIRS - len(tier1_final) - len(tier2_final)
all_queries = SAMPLE_QUERIES.copy()
all_indices = list(range(len(products)))
tier3_final = []

attempts = 0
while len(tier3_final) < n_tier3 and attempts < 5000:
    query = random.choice(all_queries)
    idx = random.randint(0, len(products) - 1)
    key = (query, idx)
    if key not in seen_pairs:
        tier3_final.append({'query': query, 'product_idx': idx,
                             'bm25_score': None, 'tier': 'T3_random'})
        seen_pairs.add(key)
    attempts += 1

print(f"Tier 3 (random): {len(tier3_final)} pairs (target {n_tier3})")
print(f"\n✅ Total pairs collected: {len(tier1_final) + len(tier2_final) + len(tier3_final)}")

Tier 1 (high score): 80 pairs (target 80)
Tier 2 (edge cases): 100 pairs (target 100)
  - Color mismatch cases: 56
Tier 3 (random): 20 pairs (target 20)

✅ Total pairs collected: 200


## Bước 7 — Ghép thông tin sản phẩm & Deduplication

In [8]:
all_pairs = tier1_final + tier2_final + tier3_final

# Ghép thông tin sản phẩm
rows = []
for pair in all_pairs:
    p = products[pair['product_idx']]
    rows.append({
        # === Thông tin cho annotator ===
        'query': pair['query'],
        'product_id': p.get('id', ''),
        'product_name': p.get('name', ''),
        'category': p.get('category', ''),
        'type': p.get('type', ''),
        'colors': ', '.join(p.get('colors', [])),
        'sizes': ', '.join(str(s) for s in p.get('sizes', [])),
        'price': p.get('price', {}).get('current', ''),
        'shop': p.get('shop', ''),
        'product_url': p.get('url', ''),
        'description_short': p.get('text_document', '')[:300] + '...',
        # === Cột gán nhãn (để trống) ===
        'label_A': '',     # Annotator A điền: exact / partial / irrelevant
        'label_B': '',     # Annotator B điền: exact / partial / irrelevant
        'note_A': '',      # Ghi chú của Annotator A
        'note_B': '',      # Ghi chú của Annotator B
        # === Metadata pipeline (ẩn khỏi annotator nếu cần) ===
        'tier': pair.get('tier', ''),
        'bm25_score': pair.get('bm25_score', ''),
        'color_mismatch': pair.get('color_mismatch', ''),
    })

# Deduplication: bỏ các cặp (query, product_id) trùng
seen_dedup = set()
deduped = []
for row in rows:
    key = (row['query'], row['product_id'])
    if key not in seen_dedup:
        deduped.append(row)
        seen_dedup.add(key)

print(f"Before dedup: {len(rows)} pairs")
print(f"After dedup:  {len(deduped)} pairs")

# Shuffle để ẩn tier order khỏi annotator
random.shuffle(deduped)

# Thêm pair_id
for i, row in enumerate(deduped):
    row['pair_id'] = f"P{i+1:03d}"

# Reorder columns — pair_id lên đầu
col_order = ['pair_id', 'query', 'product_id', 'product_name', 'category',
             'type', 'colors', 'sizes', 'price', 'shop', 'product_url',
             'description_short',
             'label_A', 'note_A', 'label_B', 'note_B',
             'tier', 'bm25_score', 'color_mismatch']

deduped = [{col: row.get(col, '') for col in col_order} for row in deduped]

print("\nSample pair:")
for k, v in list(deduped[0].items())[:10]:
    print(f"  {k}: {str(v)[:80]}")

Before dedup: 200 pairs
After dedup:  200 pairs

Sample pair:
  pair_id: P001
  query: áo quần mặc đi làm startup
  product_id: johnhenry_1073434935
  product_name: Áo Sơ Mi Nam Tay Ngắn Comfort Fit WS26SS10G-CFBB
  category: ao
  type: ao-so-mi
  colors: GREY, WHITE
  sizes: S, M, L, XL, 2XL, 3XL
  price: 700000.0
  shop: johnhenry


## Bước 8 — Kiểm tra phân bổ & Xuất Excel

In [9]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from collections import Counter

# === Kiểm tra phân bổ ===
tier_counts = Counter(r['tier'] for r in deduped)
cat_counts  = Counter(r['category'] for r in deduped)
shop_counts = Counter(r['shop'] for r in deduped)

print("=" * 50)
print("DISTRIBUTION CHECK")
print("=" * 50)
print(f"Total pairs: {len(deduped)}")
print(f"\nBy tier:")
for t, c in sorted(tier_counts.items()):
    print(f"  {t}: {c} ({c/len(deduped)*100:.0f}%)")
print(f"\nBy category:")
for t, c in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c} ({c/len(deduped)*100:.0f}%)")
print(f"\nBy shop:")
for t, c in sorted(shop_counts.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c} ({c/len(deduped)*100:.0f}%)")

# === Xuất Excel ===
OUTPUT_FILE = 'annotation_pairs.xlsx'

wb = openpyxl.Workbook()

# --- Sheet 1: Annotation (cho annotator) ---
ws1 = wb.active
ws1.title = 'Annotation'

# Columns hiển thị cho annotator (không hiện tier/metadata)
annotator_cols = ['pair_id', 'query', 'product_name', 'category', 'type',
                  'colors', 'sizes', 'price', 'shop', 'product_url',
                  'description_short', 'label_A', 'note_A']

HEADER_FILL   = PatternFill('solid', fgColor='1F4E79')
LABEL_FILL    = PatternFill('solid', fgColor='E2EFDA')
ALT_ROW_FILL  = PatternFill('solid', fgColor='F2F2F2')
HEADER_FONT   = Font(color='FFFFFF', bold=True)
CENTER        = Alignment(horizontal='center', vertical='top', wrap_text=True)
TOP_LEFT      = Alignment(horizontal='left', vertical='top', wrap_text=True)

# Header
header_labels = {
    'pair_id': 'ID', 'query': 'Query (của user)',
    'product_name': 'Tên sản phẩm', 'category': 'Category',
    'type': 'Loại', 'colors': 'Màu', 'sizes': 'Size',
    'price': 'Giá (VND)', 'shop': 'Shop',
    'product_url': 'Link sản phẩm',
    'description_short': 'Mô tả ngắn (300 ký tự)',
    'label_A': '★ NHÃN (exact/partial/irrelevant)',
    'note_A': 'Ghi chú'
}

ws1.append([header_labels.get(c, c) for c in annotator_cols])
for col_idx, col_name in enumerate(annotator_cols, 1):
    cell = ws1.cell(1, col_idx)
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = CENTER
    if col_name in ('label_A',):
        cell.fill = PatternFill('solid', fgColor='375623')

# Data rows
for row_idx, row in enumerate(deduped, 2):
    ws1.append([row.get(c, '') for c in annotator_cols])
    fill = ALT_ROW_FILL if row_idx % 2 == 0 else None
    for col_idx, col_name in enumerate(annotator_cols, 1):
        cell = ws1.cell(row_idx, col_idx)
        cell.alignment = TOP_LEFT
        if col_name == 'label_A':
            cell.fill = LABEL_FILL
        elif fill:
            cell.fill = fill

# Column widths
col_widths = {
    'pair_id': 8, 'query': 30, 'product_name': 35, 'category': 12,
    'type': 15, 'colors': 20, 'sizes': 15, 'price': 12, 'shop': 12,
    'product_url': 20, 'description_short': 50,
    'label_A': 28, 'note_A': 25
}
for col_idx, col_name in enumerate(annotator_cols, 1):
    ws1.column_dimensions[get_column_letter(col_idx)].width = col_widths.get(col_name, 15)

ws1.freeze_panes = 'A2'

# --- Sheet 2: Metadata (cho pipeline owner) ---
ws2 = wb.create_sheet('Pipeline Metadata')
meta_cols = ['pair_id', 'query', 'product_id', 'product_name',
             'tier', 'bm25_score', 'color_mismatch', 'category', 'shop']
ws2.append(meta_cols)
for col_idx in range(1, len(meta_cols)+1):
    ws2.cell(1, col_idx).fill = HEADER_FILL
    ws2.cell(1, col_idx).font = HEADER_FONT
for row in deduped:
    ws2.append([row.get(c, '') for c in meta_cols])
for col_idx, col_name in enumerate(meta_cols, 1):
    ws2.column_dimensions[get_column_letter(col_idx)].width = 20

# --- Sheet 3: Annotation Guideline ---
ws3 = wb.create_sheet('Guideline')
guideline = [
    ['ANNOTATION GUIDELINE — Query-Product Relevance'],
    [''],
    ['NHÃN', 'Định nghĩa', 'Ví dụ'],
    ['exact',
     'Sản phẩm khớp hoàn toàn với query. User tìm cái này và đây chính xác là cái đó.',
     'Query: "áo sơ mi trắng dài tay" → Sản phẩm: Áo Sơ Mi Dài Tay Trắng Solid'],
    ['partial',
     'Sản phẩm liên quan nhưng thiếu ít nhất 1 thuộc tính quan trọng trong query (màu, kiểu, dịp mặc).',
     'Query: "áo sơ mi trắng dài tay" → Sản phẩm: Áo Sơ Mi Dài Tay Xanh (đúng loại, sai màu)'],
    ['irrelevant',
     'Sản phẩm không liên quan hoặc thuộc danh mục hoàn toàn khác.',
     'Query: "áo sơ mi trắng dài tay" → Sản phẩm: Thắt lưng da nam'],
    [''],
    ['LƯU Ý QUAN TRỌNG'],
    ['1. Đánh giá từ góc nhìn của USER — họ có hài lòng khi nhận sản phẩm này không?'],
    ['2. Màu sắc: "xanh navy" ≠ "xanh lá" → irrelevant nếu màu sai hoàn toàn'],
    ['3. "đồ đi làm" có thể match áo sơ mi, quần âu, giày da → partial nếu chỉ 1 item'],
    ['4. Nếu phân vân partial vs irrelevant: ghi chú lý do vào cột Ghi chú'],
    ['5. Không nhìn vào sheet Pipeline Metadata khi đang annotate'],
]
for row in guideline:
    ws3.append(row)
ws3.cell(1, 1).font = Font(bold=True, size=14)
ws3.cell(3, 1).fill = HEADER_FILL
ws3.cell(3, 1).font = HEADER_FONT
ws3.cell(3, 2).fill = HEADER_FILL
ws3.cell(3, 2).font = HEADER_FONT
ws3.cell(3, 3).fill = HEADER_FILL
ws3.cell(3, 3).font = HEADER_FONT
ws3.column_dimensions['A'].width = 15
ws3.column_dimensions['B'].width = 60
ws3.column_dimensions['C'].width = 55
for row in ws3.iter_rows():
    for cell in row:
        cell.alignment = TOP_LEFT

wb.save(OUTPUT_FILE)
print(f"\n✅ Saved: {OUTPUT_FILE}")
print(f"   Sheet 'Annotation'        → {len(deduped)} cặp, cột label trống cho annotator")
print(f"   Sheet 'Pipeline Metadata' → thông tin tier, BM25 score")
print(f"   Sheet 'Guideline'         → hướng dẫn gán nhãn")

DISTRIBUTION CHECK
Total pairs: 200

By tier:
  T1_high: 80 (40%)
  T2_edge: 100 (50%)
  T3_random: 20 (10%)

By category:
  ao: 101 (50%)
  quan: 63 (32%)
  phu-kien: 36 (18%)

By shop:
  johnhenry: 82 (41%)
  aristino: 65 (32%)
  henstore: 28 (14%)
  icondenim: 25 (12%)

✅ Saved: annotation_pairs.xlsx
   Sheet 'Annotation'        → 200 cặp, cột label trống cho annotator
   Sheet 'Pipeline Metadata' → thông tin tier, BM25 score
   Sheet 'Guideline'         → hướng dẫn gán nhãn


## Bước 9 — Download file

In [10]:
from google.colab import files
files.download(OUTPUT_FILE)
print("✅ Download started!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


---
## Bước 10 (tùy chọn) — Tính IAA sau khi 2 annotators hoàn thành

> Upload lại file Excel đã gán nhãn (cả 2 cột `label_A` và `label_B` đã điền), chạy cell này.

In [ ]:
# Upload file đã gán nhãn xong
# from google.colab import files
# uploaded_labeled = files.upload()
# LABELED_FILE = list(uploaded_labeled.keys())[0]

# Nếu test với file hiện tại (thay label bằng dummy để demo)
LABELED_FILE = OUTPUT_FILE

import openpyxl

wb_labeled = openpyxl.load_workbook(LABELED_FILE)
ws_labeled = wb_labeled['Annotation']

header = [cell.value for cell in ws_labeled[1]]
col_label_A = header.index('★ NHÃN (exact/partial/irrelevant)') + 1
# Nếu sheet có cột label_B (cho annotator B), tìm nó
try:
    col_label_B = header.index('label_B') + 1
except ValueError:
    print("⚠️  Chưa thấy cột label_B trong sheet. Thêm cột này cho Annotator B.")
    col_label_B = None

if col_label_B:
    labels_A, labels_B = [], []
    VALID = {'exact', 'partial', 'irrelevant'}
    for row in ws_labeled.iter_rows(min_row=2, values_only=True):
        la = str(row[col_label_A - 1] or '').strip().lower()
        lb = str(row[col_label_B - 1] or '').strip().lower()
        if la in VALID and lb in VALID:
            labels_A.append(la)
            labels_B.append(lb)

    if not labels_A:
        print("⚠️  Chưa có nhãn hợp lệ nào. Hãy điền nhãn vào cột label_A và label_B trước.")
    else:
        # Percent Agreement
        agree = sum(a == b for a, b in zip(labels_A, labels_B))
        pct_agree = agree / len(labels_A)

        # Cohen's Kappa
        from sklearn.metrics import cohen_kappa_score
        kappa = cohen_kappa_score(labels_A, labels_B)

        # Confusion breakdown
        conflict_matrix = defaultdict(int)
        for a, b in zip(labels_A, labels_B):
            conflict_matrix[(a, b)] += 1

        print(f"=" * 50)
        print(f"INTER-ANNOTATOR AGREEMENT")
        print(f"=" * 50)
        print(f"Pairs evaluated: {len(labels_A)}")
        print(f"Percent Agreement: {pct_agree:.1%}")
        print(f"Cohen's Kappa:     {kappa:.3f}")
        print(f"  (0.6-0.8 = substantial, >0.8 = near-perfect)")
        print(f"\nConflict matrix (A vs B):")
        print(f"{'':12} {'exact':>10} {'partial':>10} {'irrelevant':>12}")
        for label_a in ['exact', 'partial', 'irrelevant']:
            row_str = f"A={label_a:10}"
            for label_b in ['exact', 'partial', 'irrelevant']:
                row_str += f" {conflict_matrix[(label_a, label_b)]:>10}"
            print(row_str)

        # Phân tích conflict theo tier
        print(f"\n=== Conflicts by Tier ===")
        wb2 = openpyxl.load_workbook(LABELED_FILE)
        ws_meta = wb2['Pipeline Metadata']
        meta_header = [cell.value for cell in ws_meta[1]]
        tier_col = meta_header.index('tier') + 1 if 'tier' in meta_header else None
        print("(Upload file với cả sheet Pipeline Metadata để xem conflict theo tier)")